verify ISO 8601:2004 format: YYYY-MM-DDTHH:MM:SSZ

In [1]:
import numpy as np

def iso_to_dt64(iso_str):
    ymd :  iso_str.split('T')[0]
    hms = iso_str.split('T')[1][:-1]
    dt64 = np.datetime64('T'.join([ymd, hms]))
    return dt64

# iso_str :  '2000-04-18T00:50:32Z'
# iso_str :  '2000-04-31T00:50:32Z'
# iso_str :  '0000-04-09T00:50:32Z'
# iso_str = '2000-04-18 00:50:32'
iso_str = '2007-04-15T09:26:28Z'

try:
    iso_to_dt64(iso_str)
    print('valid date format')
except:
    print('invalid date or date format')

invalid date or date format


In [2]:
def checkLicense(self):
    import json
    import re
    from urllib.parse import urlparse

    for a in self.minimal_attrs:
        if a['name'] == 'license':
            license_attr = a

    # check that license attribute is given
    if not 'license' in self.global_attrs:
        print('missing license')
        self.missing_attrs.append(license_attr)
    else:
        # only continues if license given
        # verify given license exists
        print(self.global_attrs['license'])

        # split into url and id
        print([string.replace(')', '').strip() for string in self.global_attrs['license'].split('(')])
    
        tests = [
            "http://spdx.org/licenses/CC-BY-4.0 (CC-BY-4.0)",  # both present
            "http://spdx.org/licenses/CC-BY-4.0",              # url only
            "(CC-BY-4.0)",                                      # id only
            "CC-BY-4.0",                                      # id only
            "",                                                 # nothing
        ]

        t = tests[0]
        # open JSON with licenses info
        with open('met-md-checker/data/licenses.json', 'r') as file:
            licenses = json.load(file)

        pattern = re.compile(r'^(.*?)\s*(?:\((.*)\))?\s*$')
        # for t in tests:
        m = pattern.match(t)
        # print((m.group(1).strip(), m.group(2)))
        license_url = m.group(1).strip()
        license_id  = m.group(2)

        # search license id in JSON
        for l in licenses['licenses']:
            if l['licenseId'] == license_id:
                # if license id found, check link
                print(f'license id found:\n{l['licenseId']}\n{l['reference']}')
                break


In [3]:
from urllib.parse import urlparse

tests = [
        "https://adc.met.no/",  # valid url
        "https://adc.met.no/index.html",  # valid url
        "http://adc.met.no/",   # valid url
        "",                     # nothing
        'adc.met.no',           # invalid
        'https://adc.met',      # invalid
        'www.adc.met.no',       # invalid
    ]

def is_valid_url(url):
    tokens = urlparse(url)
    return all(getattr(tokens, attr) for attr in ('scheme', 'netloc'))

for test in tests:
    try:
        print(f"{test} is valid: {is_valid_url(test)}")
    except (AttributeError, TypeError) as e:
        print(e)

https://adc.met.no/ is valid: True
https://adc.met.no/index.html is valid: True
http://adc.met.no/ is valid: True
 is valid: False
adc.met.no is valid: False
https://adc.met is valid: True
www.adc.met.no is valid: False


In [4]:
given_attrs = {
    'time_coverage_start': '2000-04-31T00:50:32Z',
    'geospatial_lon_min': '212.9',
    'publisher_url': 'adc.met.no',
    'publisher_email': 'adc.met.no',
    # 'publisher_email': 'adc@met.no',
    'publisher_name': '',
    'publisher_institution': '',
}

def _check(attr: str, validate=None) -> None:
    try:
        val = given_attrs[attr]
    except KeyError:
        print('error')

_check('title')

error


# Checks from yaml

In [5]:
%load_ext autoreload

In [6]:
import yaml
from typing import Any

# class Error(Exception):
    # pass

def load_config(config_path: str) -> dict:
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    
    if not isinstance(cfg, dict) or 'attributes' not in cfg:
        raise ConfigError('config must have "attributes" section')
    return cfg

config = load_config('mdreqs.yaml')

In [7]:
# from mdcheck import Error
CHECK_FUNCS = {}

def register_check(name: str):
    '''Decorator to register check func'''
    def decorator(func):
        CHECK_FUNCS[name] = func
        return func
    return decorator

@register_check('exists')
def check_exists(value: str):
    if value is None:
        return False
    return True

@register_check('nonempty')
def check_nonempty(value: str):
    if str(value).strip() == '':
        return False
    return True

@register_check('entries_nonempty')
def check_entries_nonempty(value: str) -> bool:
    for pair in value.split(','):
        if not pair:
            return False
    return True

@register_check('entries_prefixed')
def check_entries_prefixed(value: str) -> bool:
    for pair in value.split(','):
        if not pair:
            continue
        if ':' not in pair:
            return False
    return True

@register_check('gcmdsk_present')
def check_gcmdsk_present(value: str):
    # Separate keywords and prefixes
    kws = {}
    for pair in value.split(','):
        if not pair:
            # return False
            continue
        if ':' not in pair:
            # return False
            continue
        key, elements = pair.split(':', 1)
        key = key.strip()
        if not key:
            return False
        
        element_list = [e.strip() for e in elements.split('>') if e.strip()]
        if not element_list:
            return False
        kws.setdefault(key, []).append(element_list)

    # Verify GCMDSK prefix present
    if not 'GCMDSK' in kws:
        return False

    return True

@register_check('valid_gcmdsk')
def check_valid_gcmdsk(value: str) -> bool:
    from gcmd_tools import make_gcmd_tree
    # sktree = make_gcmd_tree('met-md-checker/data/sciencekeywords.csv')
    sktree = make_gcmd_tree('data/sciencekeywords.csv')
    # Separate keywords and prefixes
    kws = {}
    for pair in value.split(','):
        key, elements = pair.split(':', 1)
        key = key.strip()
        
        element_list = [e.strip() for e in elements.split('>') if e.strip()]
        kws.setdefault(key, []).append(element_list)
    for chain in kws['GCMDSK']:
        if not sktree.contains(*chain):
            return False
    return True

@register_check('keywords_map_to_vocabulary')
def check_keywords_map_to_vocab(values: []) -> bool:
    '''Verify that each prefix in 'keywords' is used in 'keywords_vocabulary' and vice versa'''

    # Helper function for selecting unique prefixes
    def collect_prefixes(vals: []) -> set:
        prefixes = set()
        for pair in vals.split(','):
            key, elements = pair.split(':', 1)
            key = key.strip()
            prefixes.add(key)
        return prefixes

    # Collect and compare unique prefixes
    return collect_prefixes(values[0]) == collect_prefixes(values[1])

@register_check('is_between')
def check_is_between(value: str, args: dict) -> bool:
    '''Verify that a given value is a number between min and max.'''
    try:
        val = float(value)
        return args['min'] <= val <= args['max']
    except:
        return False

@register_check('min_decimals')
def check_min_decs(value: str, args: dict) -> bool:
    '''Verify that a given number has at least specified amount of decimal points.'''
    try:
        num_decs = len(str(value).split('.')[1])
        return num_decs >= int(args['min_decs'])
    except:
        return False

@register_check('iso_8601_2004')
def check_iso_8601_2004_time_format(value: str) -> bool:
    '''Verify that ISO 8601:2004 extended date format is used, i.e. YYYY-MM-DDTHH:MM:SSZ.'''
    import re
    pattern = r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$'
    return bool(re.match(pattern, value))

@register_check('valid_date_time')
def check_valid_date_time(value: str) -> bool:
    '''Verify that datetime is valid.'''
    from utils import iso_to_dt64
    try:
        iso_to_dt64(iso_str=value)
        return True
    except ValueError: # Catches if invalid date/time -> check fails
        return False

@register_check('license_formatted_correctly')
def check_license_formatted(value: str) -> bool:
    '''Verify that license has format <URL> (<Identifier>)'''
    import re
    pattern = r'^https?://(?:www\.)?[a-zA-Z0-9][a-zA-Z0-9.-]*\.[a-zA-Z]{2,}(?:/[^\s]*)? \(([^)]+)\)$'

    # for entry in value.split(','):
    match = re.match(pattern, value.strip())
    return bool(match)

@register_check('spdx_license')
def check_spdx_license(value: str) -> bool:
    '''Verify that given license is a SPDX license'''
    import json
    import re

    # Read licenses from file
    with open('data/licenses.json', 'r') as file:
        lic_file = json.load(file)

    # Extract SPDX license IDs and URLS
    spdx_lics = {}
    for l in lic_file['licenses']:
        spdx_lics[l['licenseId']] = l['reference']

    # Extract given URL and ID
    pattern = r'^([^(]+)\s*\((.+?)\)$'
    match = re.match(pattern, value.strip())
    lic_ref = match.group(1).strip()
    lic_id  = match.group(2).strip()

    # Try to find license ID in SPDX licenses
    if spdx_lics.get(lic_id):
        # Match given URL to SPDX reference
        lic_ref = lic_ref if lic_ref.endswith('.html') else lic_ref + '.html'
        return lic_ref == spdx_lics[lic_id]
    else:
        return False

In [8]:
%autoreload
from errors import Error, MDWarning


def run_checks(config, given_attrs):
    # helper func for checking if specific test has passed
    def has_passed(ids):
        for check_id in ids:
            attr_name = check_id.split('/')[0]
            passed = next((c for c in checks_tracker[attr_name]['checks'] if c['id'] == check_id))['passed']
            if not passed:
                return False
        return True

    errors = []
    warnings = []
    # dict for tracking which checks were performed and storing the results
    checks_tracker = {}

    for attr_name, attr_config in config['attributes'].items():
        # print(attr_name, attr_config)
        value = given_attrs.get(attr_name)

        checks_tracker[attr_name] = {}
        checks_tracker[attr_name]['checks'] = []
        # print(checks_tracker[attr_name])
        # print(config['attributes'][attr_name]['checks'])

        # Execute checks
        for check in attr_config.get('checks', []):
        #     # if check_passed:
            check_type = check.get('type')
            conditions = check.get('conditions')
            args = check.get('args')

            # check_passed = False
            if conditions:
                # print(conditions, has_passed(conditions))
                if has_passed(conditions):
                    # check_passed = CHECK_FUNCS[check_type](value)
                    check_passed = CHECK_FUNCS[check_type](value, args) if args else CHECK_FUNCS[check_type](value)
                else:
                    continue
            else:
                # check_passed = CHECK_FUNCS[check_type](value)
                check_passed = CHECK_FUNCS[check_type](value, args) if args else CHECK_FUNCS[check_type](value)
            
            check_track = check.copy()
            check_track['passed'] = check_passed
            checks_tracker[attr_name]['checks'].append(check_track)
            
            if not check_passed:
                if check.get('severity') == 'warning':
                    warnings.append(MDWarning(attr=attr_name, message=check.get('message')))
                elif check.get('severity') == 'error':
                    errors.append(Error(attr=attr_name, message=check.get('message')))

    # Execute cross checks
    for check in config['cross_checks']:
        check_type = check.get('type')
        conditions = check.get('conditions')
        checks_tracker['cc'] = []


        # Execute cross check if conditions are met
        if conditions:
            if has_passed(conditions):
                attrs = check.get('involved_attributes')
                values = [given_attrs[a] for a in attrs]
                check_passed = CHECK_FUNCS[check_type](values)
            else:
                continue
        else:
            check_passed = CHECK_FUNCS[check_type](values)
        
        check_track = check.copy()
        check_track['passed'] = check_passed
        checks_tracker['cc'].append(check_track)
        
        if not check_passed:
            if check.get('severity') == 'warning':
                warnings.append(MDWarning(attr='cross-check', message=check.get('message')))
            elif check.get('severity') == 'error':
                errors.append(Error(attr='cross-check', message=check.get('message')))
    return errors, warnings

def print_report(errors, warnings):
    print(f'\n ERRORS:   {len(errors)}\n----------------')
    for e in errors:
        e.printFull()

    print(f'\n WARNINGS: {len(warnings)}\n----------------')
    for w in warnings:
        w.printFull()

In [9]:
given_attrs = {
    'title': 'very cool data set',
    'summary': ' ',
    # 'keywords': '',
    # 'keywords': 'GCMDSK:EARTH SCIENCE > OCEANS, GCMDSK:OCEANS,,fish',
    'keywords': 'GCMDSK:EARTH SCIENCE > OCEANS, GCMDSK:OCEANS',
    # 'keywords': 'GCMDSK:EARTH SCIENCE > OCEANS,OCEANS',
    # 'keywords': 'GCMDSK:EARTH SCIENCE > OCEANS',
    # 'keywords_vocabulary': 'h:url',
    'keywords_vocabulary': 'GCMDSK:url',
    # 'keywords_vocabulary': 'GCMDSK:url, test:url',
    # 'keywords_vocabulary': 'h:url,url',
    # 'keywords_vocabulary': ' ',
    # 'keywords_vocabulary': '',
    'geospatial_lat_min': '78.95738',
    # 'geospatial_lat_min': '78.9',
    'geospatial_lat_max': '78.95738',
    # 'geospatial_lat_max': '278.95738',
    # 'geospatial_lat_min': '278.95738',
    'geospatial_lon_min': '-180.00',
    'geospatial_lon_max': '-0.89',

    'time_coverage_start': '2000-04-18T00:40:43Z',
    'time_coverage_end': '2000-04-18T00:40:43Z',
    # 'time_coverage_start': '2000-04-18T00:40:43',
    # 'time_coverage_start': '2000-04-31T00:40:43Z',
    # 'time_coverage_start': '2000-04-31T00:60:43Z',
    # 'time_coverage_start': '18.04.2000',
    'date_created': '2004-05-22T12:50:20Z',
    # 'license': 'https://www.myurl.com (0BSD)',
    # 'license': 'https://spdx.org/licenses/0BSD.html (0BSD)',
    'license': 'https://spdx.org/licenses/0BSD.html (0BSD)',
}

errors, warnings = run_checks(config=config, given_attrs=given_attrs)
print_report(errors, warnings)


 ERRORS:   1
----------------
Error: keywords contains an invalid GCMD science keyword.
Include the full path of valid keywords formatted like "GCMDSK:[category] > [topic] > [...]".


 WARNINGS: 1
----------------
